# 02 — Analysis

Persist the cross-ecosystem match table using the project's columnar container
(`<table>/<column>.zst.partNN` inside an **uncompressed** tar), verify the
round-trip, and confirm the FAT32 constraint holds.

In [ ]:
import json
import shutil
import sys
import tarfile
from pathlib import Path

sys.path.insert(0, str(Path.cwd() / "src"))

from msys2_dataset.arch import ArchPackage
from msys2_dataset.matching import match_ecosystems
from msys2_dataset.packages import parse_ls_listing_file
from msys2_dataset.store import FAT32_MAX_BYTES, read_table, write_table

FAT32_MAX_BYTES, FAT32_MAX_BYTES / 1024**3

### Build the table from the three ecosystems

In [ ]:
arch_path = Path("data/arch-pkgs.jsonl")
if arch_path.is_file():
    arch = [
        ArchPackage(**json.loads(line))
        for line in arch_path.read_text(encoding="utf-8").splitlines()
        if line.strip()
    ]
else:
    from msys2_dataset.arch import fetch_official_repo

    arch = fetch_official_repo("core")

matches = match_ecosystems(
    arch,
    parse_ls_listing_file("data/msys2-pkgs.tsv"),
    parse_ls_listing_file("data/mingw-pkgs.tsv"),
)

table = {
    "name": [m.name for m in matches],
    "in_arch": [m.in_arch for m in matches],
    "in_msys2": [m.in_msys2 for m in matches],
    "in_mingw": [m.in_mingw for m in matches],
    "ecosystems": [",".join(m.ecosystems) for m in matches],
    "arch_names": [",".join(p.name for p in m.arch) for m in matches],
    "arch_repos": [",".join(sorted({p.repo for p in m.arch})) for m in matches],
    "msys2_names": [",".join(p.name for p in m.msys2) for m in matches],
    "mingw_names": [",".join(p.name for p in m.mingw) for m in matches],
}
{column: len(values) for column, values in table.items()}

### Write it as a columnar zstd archive

In [ ]:
# Build into a scratch directory so running the notebook does not dirty the
# tracked tree; the real pipeline writes to data/processed/.
out = Path(".notebook-tmp/processed")
out.mkdir(parents=True, exist_ok=True)
archive = out / "matches.tar"
write_table(archive, "matches", table, max_part_bytes=1024 * 1024)
archive.stat().st_size

### Inspect the produced layout

In [ ]:
import tarfile

with tarfile.open(archive, mode="r:") as tar:  # "r:" rejects compression
    names = tar.getnames()
print(f"{len(names)} members")
names[:8]

### Verify the round-trip and the size cap

In [ ]:
import json

# Extract into a separate directory; deleting the parent would remove the
# archive we are reading from.
extract_dir = Path(".notebook-tmp/extracted")
shutil.rmtree(extract_dir, ignore_errors=True)
restored = read_table(archive, extract_dir)
assert restored == table, "round-trip mismatch"
print("round-trip OK")

with tarfile.open(archive, mode="r:") as tar:
    oversize = [m.name for m in tar.getmembers() if m.size > FAT32_MAX_BYTES]
assert not oversize, oversize
print("every member is within the FAT32 limit")

### SQLite alternative

For shards that are rewritten repeatedly, the SQLite path is faster than
re-serialising a whole column. SQLite columns are scalar, so the list-valued
fields are stored as JSON strings.

In [ ]:
from msys2_dataset.sqlite_store import load_rows, write_rows

db = out / "matches.sqlite"
matched = [m for m in matches if m.in_arch and (m.in_msys2 or m.in_mingw)]
records = [
    {
        "name": m.name,
        "in_msys2": int(m.in_msys2),
        "in_mingw": int(m.in_mingw),
        "ecosystems": ",".join(m.ecosystems),
        "arch_names": json.dumps([p.name for p in m.arch]),
        "msys2_names": json.dumps([p.name for p in m.msys2]),
        "mingw_names": json.dumps([p.name for p in m.mingw]),
    }
    for m in matched
]
write_rows(db, "matches", records, if_exists="replace")
rows = load_rows(db, "matches")
print(f"{len(rows)} matched rows in {db}")
rows[0]